In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import fitsio
from pycorr import TwoPointCorrelationFunction, TwoPointEstimator, project_to_multipoles, project_to_wp, utils, setup_logging
from scipy.optimize import curve_fit
from astropy.table import Table
from astropy.io import fits

from dataloc import *
from desiclusteringtools import *
import plotting as pp
from clusteringtools import *

# MAKE ALL PLOTS TEXT BIGGER
plt.rcParams.update({'font.size': 15})
# But legend a bit smaller
plt.rcParams.update({'legend.fontsize': 12})
# Set DPI up a bit
plt.rcParams.update({'figure.dpi': 150})

# This notebook is for DESI Project 712

### Maps

In [ ]:
rpath = '/global/cfs/cdirs/desi/users/ianw89/groupcatalogs/BGS_Y3/v0.6/BGS_BRIGHT_17_full.ran.fits'
tbl = Table(fitsio.read(rpath))
filepath = '/global/cfs/cdirs/desi/users/ianw89/groupcatalogs/BGS_Y3/v0.6/GROUP_CATALOG_BGS_Y3_v0.6.fits'
groupcat = Table.read(filepath)
print(len(tbl))

In [ ]:
# Downsample to 10%
np.random.seed(42)
sel = np.random.rand(len(tbl)) < 0.033
tbl_s = tbl[sel]
sel2 = np.random.rand(len(groupcat)) < 0.1
groupcat_s = groupcat[sel2]

In [ ]:
#make map
f=pp.make_map(tbl_s['RA'].value, tbl_s['DEC'].value)
f=pp.make_map(groupcat_s['RA'].value, groupcat_s['DEC'].value)

### Get Early Sersic Test Results

In [ ]:
# Define the base directory where your clustering results are stored.
clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/newclustering/DA2/LSS/loa-v1/LSScats/v1.1/'
all_results = load_allcounts_from_disk(clustering_base_dir)

# You can now access the data and parameters like this:
if all_results:
    print("\nExample of first loaded result:")
    first_result = all_results[0]
    print("Parameters:", first_result['params'])
    print("Data object:", first_result['data'])



In [ ]:
# Now, call the function with the loaded results and a specific weight type
plot_wp_QSF_bins(all_results, 'pip_bitwise')

In [ ]:
# Call the new function to generate the comparison plots
plot_weight_comparison(all_results)

Imaging Systematics Check

In [ ]:
# Find all *_linclusimsysfit.png files in the clustering_base_dir and its subdirectories
import glob
linclusimsysfit_files = glob.glob(os.path.join(clustering_base_dir, '**', '*_linclusimsysfit.png'), recursive=True) 
print(f"Found {len(linclusimsysfit_files)} linclusimsysfit.png files:")
for f in linclusimsysfit_files:
    print(f" - {f}")

In [ ]:
# First linclusimsysfit_files investigation
for i in range(10):
    first_file = linclusimsysfit_files[i]
    plt.figure(figsize=(10, 10))
    img = plt.imread(first_file)
    plt.imshow(img)
    plt.axis('off') 
    plt.show()

### Get the mag / g-r clustering test clustering

In [ ]:
# Define the base directory where your clustering results are stored.
#clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/v0.1'
clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/v0.2'
all_results = load_allcounts_from_disk(clustering_base_dir)

In [ ]:
# Find the dataset we will use as the reference
reference_result = None # Primary reference data set
downsample_tests = [] # List to hold downsampled test results
footprint_tests = [] # List to hold footprint test results
for result in all_results:
    mag_range = result['params'].get('mag_range')
    gmr_range = result['params'].get('gr_range')  
    njack = result['params'].get('njack')
    if mag_range is None and gmr_range is None and int(njack) == 64 and result['params'].get('region') == 'GCcomb':
        if result['params'].get('downsample') is None and result['params'].get('nran') != 1:
            reference_result = result
        downsample_tests.append(result)
    if mag_range is None and gmr_range is None and int(njack) == 64 and result['params'].get('downsample') is None and int(result['params'].get('nran')) != 1:
        footprint_tests.append(result)

# TODO for footprint tests - how does jackknife work for SGC vs NGC vs GCcomb?

ref_estimator = reference_result['data']
rp_ref, wp_ref, cov_ref = ref_estimator.get_corr(return_sep=True, return_cov=True, mode='wp')

In [ ]:
# Filter to chosen weighting scheme
filtered_results = [res for res in all_results if res['params']['weights'] == 'pip_bitwise']

# Filter to chosen number of jackknife regions
filtered_results = [res for res in filtered_results if int(res['params'].get('njack')) == 64]

# Filter to GCcomb region
filtered_results = [res for res in filtered_results if res['params'].get('region') == 'GCcomb']

print(f"Kept {len(filtered_results)} results.")

In [ ]:
# Group results by magnitude range
results_by_mag = {}
for result in filtered_results:
    mag_range = result['params'].get('mag_range')
    if mag_range is None:
        continue
    if mag_range not in results_by_mag:
        results_by_mag[mag_range] = []
    results_by_mag[mag_range].append(result)

# Order by magnitude range 
results_by_mag = dict(sorted(results_by_mag.items(), key=lambda x: float(x[0].split('to')[0])))

#assert len(results_by_mag) == 10

In [ ]:
plot_wp_mag_gmr(results_by_mag, reference_result)

In [ ]:
# TwoPointEstimator doesn't have the count of galaxies in the sample. Let's get it from disk
for mag_range, measurements in results_by_mag.items():
    for measurement in measurements:
        testpath = clustering_base_dir + f'/BGS_BRIGHT_CEN_mag{mag_range}_gr{measurement["params"].get("gr_range")}_clustering.dat.fits'
        # Get the number of rows without reading everything in
        with fits.open(testpath) as hdul:
            nrows = hdul[1].header['NAXIS2']
        measurement['params']['n_galaxies'] = nrows

In [ ]:
def display_cov(measurement):
    mag_range = measurement['params'].get('mag_range')
    rp, wp, cov = measurement['data'].get_corr(return_sep=True, return_cov=True, mode='wp')
    plt.figure()
    plt.title(f"Mag range: {mag_range}, g-r: {measurement['params'].get('gr_range')}")
    # Use rp as x and y axis
    plt.xticks(ticks=np.arange(len(rp)), labels=[f"{r:.1f}" for r in rp], rotation=45)
    plt.yticks(ticks=np.arange(len(rp)), labels=[f"{r:.1f}" for r in rp])
    plt.imshow(cov, cmap='viridis', aspect='auto')
    plt.colorbar(label='Covariance')
    plt.show()

def display_corr(measurement):
    mag_range = measurement['params'].get('mag_range')
    region = measurement['params'].get('region')
    rp, wp, cov = measurement['data'].get_corr(return_sep=True, return_cov=True, mode='wp')
    corr = cov / np.outer(np.sqrt(np.diag(cov)), np.sqrt(np.diag(cov)))
    plt.figure()
    if mag_range is not None:
        plt.title(f"Mag range: {mag_range}, g-r: {measurement['params'].get('gr_range')}")
    elif measurement['params'].get('downsample') is not None:
        plt.title(f"Downsampled to 1/{measurement['params'].get('downsample')} of reference")
    elif region != 'GCcomb':
        plt.title(f"Region: {region}")
    else:
        plt.title(f"Full reference sample")
    # Use rp as x and y axis
    plt.xticks(ticks=np.arange(len(rp)), labels=[f"{r:.1f}" for r in rp], rotation=45)
    plt.yticks(ticks=np.arange(len(rp)), labels=[f"{r:.1f}" for r in rp])
    plt.imshow(corr, cmap='viridis', aspect='auto', vmin=-1, vmax=1)
    plt.colorbar(label='Correlation')
    plt.show()

#for mag_range, measurements in results_by_mag.items():
#    for measurement in measurements:
#        display_corr(measurement)

In [ ]:
# Let's see how the correlation matrices evolve as we downsample the reference.
for downsample_result in downsample_tests:
    display_corr(downsample_result)

In [ ]:
# Let's see how the correlation matrices evolve for different footprints
for result in footprint_tests:
    display_corr(result)

In [ ]:
# TODO density instead of number
# TODO volume instead of number
# Let's plot the mean off diagonal element of the correlation matrix as a function of the number of galaxies in the sample
for measurement in downsample_tests:
    corr = measurement['data'].get_corr(return_sep=True, return_cov=True, mode='wp')[2]
    corr = corr / np.outer(np.sqrt(np.diag(corr)), np.sqrt(np.diag(corr)))
    off_diag_elements = corr[~np.eye(corr.shape[0], dtype=bool)]
    mean_off_diag = np.mean(off_diag_elements)
    measurement['mean_off_diag'] = mean_off_diag

plt.figure(figsize=(10, 6))

for mag_range, measurements in results_by_mag.items():
    n_galaxies = []
    mean_off_diag = []
    n_galaxies.extend([m['params']['n_galaxies'] for m in measurements])
    mean_off_diag.extend([m['mean_off_diag'] for m in measurements])
    plt.scatter(n_galaxies, mean_off_diag, marker='o', label=f"{mag_range}")

plt.xlabel("Number of galaxies in sample")
plt.ylabel("Mean off-diagonal correlation")
plt.legend()

In [ ]:
# Let's plot the mean off diagonal element of the correlation matrix as a functino of the number of galaxies in the sample
for mag_range, measurements in results_by_mag.items():
    for measurement in measurements:
        corr = measurement['data'].get_corr(return_sep=True, return_cov=True, mode='wp')[2]
        corr = corr / np.outer(np.sqrt(np.diag(corr)), np.sqrt(np.diag(corr)))
        off_diag_elements = corr[~np.eye(corr.shape[0], dtype=bool)]
        mean_off_diag = np.mean(off_diag_elements)
        measurement['mean_off_diag'] = mean_off_diag

plt.figure(figsize=(10, 6))

for mag_range, measurements in results_by_mag.items():
    n_galaxies = []
    mean_off_diag = []
    n_galaxies.extend([m['params']['n_galaxies'] for m in measurements])
    mean_off_diag.extend([m['mean_off_diag'] for m in measurements])
    plt.scatter(n_galaxies, mean_off_diag, marker='o', label=f"{mag_range}")

plt.xlabel("Number of galaxies in sample")
plt.ylabel("Mean off-diagonal correlation")
plt.legend()

In [ ]:
# Let's look at the actual raw number of pair counts in an rp bin for data and randoms to get a feel for things
for mag_range, measurements in results_by_mag.items():
    print("Mag range:", mag_range)
    for measurement in measurements:
        estimator = measurement['data']
        #print(estimator.__dir__())
        #print(estimator.edges)
        # rp bin 2 (~3.3 Mpc/h), z bin 40 (near 0)
        # Let's sum over z
        print(estimator.D1D2.ncounts[2].sum())
        break
    break

In [ ]:
# Calculate bias b compared to the reference
for mag_range, measurements in results_by_mag.items():
    for measurement in measurements:
        rp, wp, cov = measurement['data'].get_corr(return_sep=True, return_cov=True, mode='wp')
        bias_info = get_bias((rp_ref, wp_ref, cov_ref), (rp, wp, cov))
        measurement['bias'] = bias_info[0]
        measurement['bias_err_up'] = bias_info[1]
        measurement['bias_err_down'] = bias_info[2]

In [ ]:
# 2 rows of 5 plots. One for each mag group. Plot the bias values for each color 
fig, axes = plt.subplots(2, 5, figsize=(17, 7), sharex=True, sharey=True)
axes = axes.flatten()
i = 0
for mag_range, measurements in results_by_mag.items():
    ax = axes[i]
    i += 1

    ax.errorbar(
        [np.mean([float(x) for x in m['params'].get('gr_range', '0to0').split('to')]) for m in measurements],
        [m['bias'] for m in measurements],
        yerr=[[m['bias_err_down'] for m in measurements], [m['bias_err_up'] for m in measurements]],
        fmt='.',
        capsize=3,
    )

    #ax.set_xlim([0.2, 1.1])
    ax.set_ylim([0.1, 2.5])  
    ax.set_xlabel('g-r')
    ax.set_ylabel('b')
    title = f'{float(mag_range.split("to")[0]):.1f} to {float(mag_range.split("to")[1]):.1f}'
    ax.set_title(title)
    
plt.tight_layout()

In [ ]:
# TODO Worry. 
# I am getting different bias values (not just errors) when I either don't use target cov matrix vs use cov with 4 jackknife regions vs 64 jackknife regions.

In [ ]:
# SAVE RESULTS for use outside NERSC ecosystem
outdir = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/v0.2/export/'
save_wp_for_maggmr(outdir, filtered_results)

save_wp_dr2format(os.path.join(outdir, 'wp_reference.dat'), (rp_ref, wp_ref, cov_ref))

In [ ]:
# Ensure roundtrip
test_rp, test_wp, test_cov = load_wp_dr2format(os.path.join(outdir, 'wp_reference.dat'))
assert np.allclose(test_rp, rp_ref)
assert np.allclose(test_wp, wp_ref)
assert np.allclose(test_cov, cov_ref)